# Source Model Statistics Explorer

This notebook browses remote ESHM20 files within a chosen dataset version scope and then analyzes a selected supported subset.

Discovery is intentionally broader than analysis: the inventory view shows directories plus supported and unsupported files under the chosen scope, while the current v1 analysis path remains limited to `asm_v12e` area-source XML files.


Reusable implementation logic lives in `notebook_support/source_model_statistics_explorer_helpers.py`.

Use that module to inspect parsing and filtering details. This notebook stays focused on analysis choices and figure construction.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

from notebook_support.source_model_statistics_explorer_helpers import (
    REPO_RAW_BASE,
    build_truncgr_mfd,
    closed_polygon,
    compute_bounds_from_docs,
    describe_selected_files,
    discover_scoped_inventory,
    discover_version_roots,
    filter_sources_by_region,
    load_selected_source_model_files,
    normalize_region_selection,
    parse_polygon_text,
    summarize_source_inventory,
    validate_selected_files,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 10, 'legend.fontsize': 8})
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', None)


## 1. Choose Scope, Then Choose Files

Start by listing version roots under `oq_computational/` for a repository ref. Then choose a visible `version_root` and `discovery_subpath`, inspect the discovered inventory, choose an analysis-supported family from that output, and finally select files explicitly from the supported table.

The v1 analysis path remains intentionally narrow: `asm_v12e` area-source XML files (with known blocked variants excluded).


In [ ]:
# Step 1: choose a repository ref and discover version roots.
repo_ref = 'master'

version_roots = discover_version_roots(repo_ref=repo_ref)
if version_roots.empty:
    raise ValueError('No version-root candidates discovered under oq_computational/.')

print(f"Discovered version roots at ref '{repo_ref}': {len(version_roots)}")
display(version_roots[['name', 'repo_rel', 'version_tag', 'region']])


Step 2 in the next cell is the editable scope choice. Copy `version_root` from the Step 1 table, set `discovery_subpath`, then run to discover the inventory for that scope.

In [ ]:
## Steps 2 and 3:

# ===> Step 2: choose one visible version root and a discovery subpath from the table above.
version_root = 'oq_computational/oq_configuration_eshm20_v12e_region_main'
discovery_subpath = 'source_models'  # Set '' to browse the full version root.

if version_root not in set(version_roots['repo_rel']):
    raise ValueError(
        'version_root must be one of the discovered repo_rel values. '
        f"Current value: {version_root}"
    )

# ===> Step 3: discover inventory for the selected scope and inspect what exists there.
discovery_result = discover_scoped_inventory(
    repo_ref=repo_ref,
    version_root=version_root,
    discovery_subpath=discovery_subpath,
)

scope = discovery_result['scope']
inventory = discovery_result['inventory']
supported_inventory = discovery_result['supported_inventory']
summary = discovery_result['summary']
warnings = discovery_result['warnings']

print('Current discovery scope:')
print(f"- repo_ref: {scope['repo_ref']}")
print(f"- version_root: {scope['version_root']}")
print(f"- discovery_subpath: {scope['discovery_subpath'] or '(entire version root)'}")
print(f"- scope_root: {scope['scope_root']}")
print('')
print(f"Discovered rows: {summary['n_inventory_rows']}")
print(f"Supported rows: {summary['n_supported_rows']}")
if warnings:
    print('Warnings:')
    for warning in warnings:
        print(f'- {warning}')

print('')
print('Entries visible in this scope (directories/files):')
display(summary['counts_by_entry_type'])

inventory_view_columns = [
    'relative_path',
    'entry_type',
    'extension',
    'family_dir',
    'analysis_status',
    'name',
]
inventory_view = inventory[inventory_view_columns]
preview_limit = 40

print(f"Compact inventory preview: first {min(len(inventory_view), preview_limit)} of {len(inventory_view)} rows")
display(inventory_view.head(preview_limit))
if len(inventory_view) > preview_limit:
    print('Use `inventory[inventory_view_columns]` to inspect the full discovered inventory when needed.')


The discovered scope can contain multiple model families and mixed file types. Not every discovered family is currently analyzable in this notebook.

The next tables bridge broad discovery to the current analysis path by showing which families are present, how many rows each contributes, and which entries are analysis-supported versus only visible during discovery.


In [ ]:
# Step 4: inspect family and support-status visibility before choosing a family.
family_counts = (
    inventory.groupby('family_dir', dropna=False)
    .size()
    .rename('rows_in_scope')
    .reset_index()
    .sort_values('rows_in_scope', ascending=False)
)

support_counts = (
    inventory.groupby('analysis_status', dropna=False)
    .size()
    .rename('rows_in_scope')
    .reset_index()
    .sort_values('rows_in_scope', ascending=False)
)

family_support_breakdown = (
    inventory.pivot_table(
        index='family_dir',
        columns='analysis_status',
        values='relative_path',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

status_columns = [col for col in family_support_breakdown.columns if col != 'family_dir']
family_support_breakdown['rows_in_scope'] = family_support_breakdown[status_columns].sum(axis=1)
family_support_breakdown = family_support_breakdown.sort_values('rows_in_scope', ascending=False)

print('Model families visible in this scope:')
display(family_counts)

print('Support status of discovered entries in this scope:')
display(support_counts)

print('Family/support breakdown used to choose the analysis path:')
display(family_support_breakdown)


In [ ]:
# Step 5: choose the supported analysis family shown above.
dataset_family = 'asm_v12e'

available_family_dirs = sorted(inventory['family_dir'].dropna().unique())
if dataset_family not in set(available_family_dirs):
    raise ValueError(
        'dataset_family must be chosen from discovered family_dir values. '
        f"Current value: {dataset_family}. Available: {available_family_dirs}"
    )

print(f"Chosen analysis family: {dataset_family}")
print('The current v1 analysis path continues with supported asm_v12e XML files.')


In [ ]:
# Step 6: inspect files that are analysis-supported for the chosen family.
supported_family_inventory = (
    supported_inventory[supported_inventory['family_dir'] == dataset_family]
    .copy()
    .sort_values('repo_rel')
)

if supported_family_inventory.empty:
    raise ValueError(
        f'No analysis-supported rows found for dataset_family={dataset_family!r} in the selected scope.'
    )

supported_selection_table = supported_family_inventory[['repo_rel']].copy().reset_index(drop=True)
supported_selection_table.insert(0, 'selection_id', supported_selection_table.index + 1)
supported_selection_table['file'] = supported_selection_table['repo_rel'].str.split('/').str[-1]
supported_selection_table = supported_selection_table[['selection_id', 'file', 'repo_rel']]

print(f"Supported files for dataset_family='{dataset_family}': {len(supported_selection_table)}")
print('Copy repo_rel values from the table below into selected_files in the next cell.')
with pd.option_context('display.max_colwidth', None, 'display.width', 240):
    display(supported_selection_table)


### Step 7. Choose Files In The Next Cell

The next code cell is the main place where files are chosen for analysis.

Copy exact `repo_rel` values from the supported-file table shown just above.

The notebook will only analyze files listed in `selected_files`, and then immediately render a whole-domain source map for the chosen set.


In [ ]:
# Step 7: explicitly choose repo_rel values from the table above.
selected_files = [
    'oq_computational/oq_configuration_eshm20_v12e_region_main/source_models/asm_v12e/asm_ver12e_winGT_fs017_hi_abgrs_maxmag_low.xml',
    'oq_computational/oq_configuration_eshm20_v12e_region_main/source_models/asm_v12e/asm_ver12e_winGT_fs017_hi_abgrs_maxmag_mid.xml',
    'oq_computational/oq_configuration_eshm20_v12e_region_main/source_models/asm_v12e/asm_ver12e_winGT_fs017_hi_abgrs_maxmag_upp.xml',
]

selected_files = validate_selected_files(selected_files=selected_files, dataset_family=dataset_family)

supported_repo_rels = set(supported_family_inventory['repo_rel'])
out_of_scope = [repo_rel for repo_rel in selected_files if repo_rel not in supported_repo_rels]
if out_of_scope:
    raise ValueError(
        'selected_files must be chosen from supported files for the chosen dataset_family:\n - '
        + '\n - '.join(out_of_scope)
    )

selected_file_table = describe_selected_files(selected_files=selected_files, dataset_family=dataset_family)

print('')
print(f'Remote raw base: {REPO_RAW_BASE}')
print(f'Selected file count: {len(selected_files)}')
print('Validated selected files:')
display(selected_file_table[['file', 'repo_rel', 'dataset_family', 'parser_path']])


## 2. First visual check: where are the selected sources located?

After file validation, this is the first scientific orientation view. It shows the full-domain source footprints of the selected files before any regional filtering.


The next cell loads the selected files and renders the whole-domain map using a consistent color per file.

This gives an immediate spatial sanity check before moving into deeper inventory/statistics summaries.


In [ ]:
file_labels = {repo_rel: repo_rel.split('/')[-1] for repo_rel in selected_files}
color_cycle = plt.get_cmap('tab10').colors
file_colors = {repo_rel: color_cycle[i % len(color_cycle)] for i, repo_rel in enumerate(selected_files)}

parsed_docs = load_selected_source_model_files(
    selected_files=selected_files,
    dataset_family=dataset_family,
    repo_raw_base=REPO_RAW_BASE,
)

whole_map_figsize = (9.0, 6.5)
whole_map_linewidth = 0.9
whole_map_alpha = 0.55

fig, ax = plt.subplots(figsize=whole_map_figsize)

for repo_rel in selected_files:
    first = True
    for source in parsed_docs[repo_rel]['sources']:
        polygon = closed_polygon(source['polygon_lonlat'])
        if polygon is None:
            continue
        xs = [point[0] for point in polygon]
        ys = [point[1] for point in polygon]
        ax.plot(
            xs,
            ys,
            color=file_colors[repo_rel],
            linewidth=whole_map_linewidth,
            alpha=whole_map_alpha,
            label=file_labels[repo_rel] if first else None,
        )
        first = False

bounds = compute_bounds_from_docs(parsed_docs)
if bounds is not None:
    min_lon, max_lon, min_lat, max_lat = bounds
    pad_lon = max(0.3, 0.03 * (max_lon - min_lon))
    pad_lat = max(0.3, 0.03 * (max_lat - min_lat))
    ax.set_xlim(min_lon - pad_lon, max_lon + pad_lon)
    ax.set_ylim(min_lat - pad_lat, max_lat + pad_lat)

ax.set_title('Whole-domain source footprints for selected files')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, linewidth=0.35, alpha=0.3)
ax.legend(loc='best', frameon=True)
plt.tight_layout()


## 3. The files' contents

With the spatial context established, this section gives a compact file-level source inventory summary before region filtering and parameter-level analysis.


In [ ]:
inventory_summary = summarize_source_inventory(parsed_docs)
display(inventory_summary[['file', 'n_area_sources', 'n_trunc_gr', 'n_incremental']])


## 4. What changes inside a selected region?

This step applies an explicit region selection and shows which sources are retained. The operation changes source membership only and leaves source rates unchanged.

Edit `region_selection` in the next cell, then run the following cell to apply filtering and preview retained polygons.


In [ ]:
# Editable region selection state (main v1 modes: 'whole' and 'circle').
region_selection = {
    'region_mode': 'circle',
    'circle_lat': 45.0,
    'circle_lon': 12.0,
    'circle_radius_km': 300.0,
}

# Advanced hook (optional): polygon mode is available but not the main v1 path.
# region_selection = {
#     'region_mode': 'polygon',
#     'polygon_text': '11.0,44.0; 15.0,44.0; 15.0,47.0; 11.0,47.0',
# }


print('Current region selection state:')
print(region_selection)
print('')


The helper applies source-membership filtering and returns `filtered_docs` with `filter_summary`.

You can change the region definition above and tune `region_map_figsize`, line widths, and alpha values for the preview.


In [ ]:
region_selection = normalize_region_selection(region_selection)
filtered_docs, filter_summary = filter_sources_by_region(parsed_docs, region_selection)

display(filter_summary[['file', 'sources_before', 'sources_after_filter', 'sources_removed', 'pct_kept']])
print('Filtering selects sources only; it does not modify source rates.')

region_map_figsize = (9.0, 6.5)
muted_linewidth = 0.8
retained_linewidth = 1.2
retained_fill_alpha = 0.12

fig, ax = plt.subplots(figsize=region_map_figsize)

for doc in parsed_docs.values():
    for source in doc['sources']:
        polygon = closed_polygon(source['polygon_lonlat'])
        if polygon is None:
            continue
        xs = [point[0] for point in polygon]
        ys = [point[1] for point in polygon]
        ax.plot(xs, ys, color='0.75', linewidth=muted_linewidth, alpha=0.4)

for repo_rel in selected_files:
    for source in filtered_docs[repo_rel]['sources']:
        polygon = closed_polygon(source['polygon_lonlat'])
        if polygon is None:
            continue
        xs = [point[0] for point in polygon]
        ys = [point[1] for point in polygon]
        ax.fill(xs, ys, color=file_colors[repo_rel], alpha=retained_fill_alpha)
        ax.plot(xs, ys, color=file_colors[repo_rel], linewidth=retained_linewidth, alpha=0.95)

if region_selection['region_mode'] == 'circle':
    lat0 = region_selection['circle_lat']
    lon0 = region_selection['circle_lon']
    radius_km = region_selection['circle_radius_km']
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(lat0))
    dlon = radius_km / max(km_per_deg_lon, 1e-6)
    dlat = radius_km / km_per_deg_lat
    theta = np.linspace(0.0, 2.0 * np.pi, 240)
    ax.plot(lon0 + dlon * np.cos(theta), lat0 + dlat * np.sin(theta), color='black', linestyle='--', linewidth=1.8)
    ax.scatter([lon0], [lat0], color='black', s=22, zorder=5)

if region_selection['region_mode'] == 'polygon':
    region_poly = closed_polygon(parse_polygon_text(region_selection['polygon_text']))
    if region_poly is not None:
        ax.plot([p[0] for p in region_poly], [p[1] for p in region_poly], color='black', linestyle='--', linewidth=1.8)

bounds = compute_bounds_from_docs(parsed_docs)
if bounds is not None:
    min_lon, max_lon, min_lat, max_lat = bounds
    pad_lon = max(0.3, 0.03 * (max_lon - min_lon))
    pad_lat = max(0.3, 0.03 * (max_lat - min_lat))
    ax.set_xlim(min_lon - pad_lon, max_lon + pad_lon)
    ax.set_ylim(min_lat - pad_lat, max_lat + pad_lat)

legend_handles = [Line2D([0], [0], color='0.75', lw=1.0, label='All source polygons (muted)')]
for repo_rel in selected_files:
    legend_handles.append(Line2D([0], [0], color=file_colors[repo_rel], lw=2.0, label=f"Retained: {file_labels[repo_rel]}"))
if region_selection['region_mode'] == 'circle':
    legend_handles.append(Line2D([0], [0], color='black', lw=1.8, linestyle='--', label='Selected circle'))

ax.set_title(f"Region preview ({region_selection['region_mode']}): muted = all, colored = retained")
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, linewidth=0.35, alpha=0.3)
ax.legend(handles=legend_handles, loc='best', frameon=True)
plt.tight_layout()


## 5. Scientific walkthrough after region filtering

After the region is set, we read the results in a fixed order: what remains, how much rate is present, how that rate is distributed by magnitude, and which parameter patterns explain the differences.


### What remains after filtering?

Before comparing rates, we first check retained source support in each file. If one file keeps many more sources inside the region, that alone can shift aggregated MFD levels.


In [ ]:
mfd_bin_width = 0.1
mfd_bundle = build_truncgr_mfd(filtered_docs, bin_width=mfd_bin_width)
trunc_table = mfd_bundle['trunc_table'].copy()

retained_trunc_counts = (
    trunc_table.groupby('repo_rel').size().rename('n_trunc_gr_after_filter')
    if not trunc_table.empty
    else pd.Series(dtype='int64', name='n_trunc_gr_after_filter')
)

retained_baseline = (
    pd.DataFrame({'repo_rel': selected_files})
    .merge(
        filter_summary[['repo_rel', 'sources_before', 'sources_after_filter', 'pct_kept']],
        on='repo_rel',
        how='left',
    )
    .merge(retained_trunc_counts.reset_index(), on='repo_rel', how='left')
)

retained_baseline['file'] = [file_labels[repo_rel] for repo_rel in retained_baseline['repo_rel']]
retained_baseline['n_trunc_gr_after_filter'] = retained_baseline['n_trunc_gr_after_filter'].fillna(0).astype(int)
retained_baseline['pct_kept'] = retained_baseline['pct_kept'].round(1)

display(
    retained_baseline[
        ['file', 'sources_before', 'sources_after_filter', 'pct_kept', 'n_trunc_gr_after_filter']
    ]
)



### How much seismicity is present at key magnitudes?

A small threshold table gives the fastest rate-level comparison. The thresholds are chosen from the magnitude support shared by all selected files so the rows remain directly comparable.


In [ ]:
if trunc_table.empty:
    print('No truncated-GR sources are available under the current selection.')
else:
    per_file_range = trunc_table.groupby('repo_rel').agg(
        min_mag=('minMag', 'min'),
        max_mag=('maxMag', 'max'),
    )
    common_min = float(np.ceil(per_file_range['min_mag'].max() / mfd_bin_width) * mfd_bin_width)
    common_max = float(np.floor(per_file_range['max_mag'].min() / mfd_bin_width) * mfd_bin_width)

    preferred_thresholds = [5.0, 6.0, 7.0]
    thresholds = [m for m in preferred_thresholds if common_min <= m <= common_max]

    if len(thresholds) < 2:
        if common_max > common_min:
            thresholds = sorted({round(float(v), 1) for v in np.linspace(common_min, common_max, num=3)})
        else:
            thresholds = [round(common_min, 1)]

    thresholds = thresholds[:3]
    mag_starts = np.asarray(mfd_bundle['mag_bin_starts'], dtype=float)

    threshold_rows = []
    for repo_rel in selected_files:
        stats = mfd_bundle['per_file'].get(repo_rel, {})
        cumulative = np.asarray(stats.get('cumulative', np.zeros_like(mag_starts)), dtype=float)
        row = {'file': file_labels[repo_rel]}

        for threshold in thresholds:
            idx = int(np.searchsorted(mag_starts, threshold, side='left'))
            rate = float(cumulative[idx]) if idx < cumulative.size else 0.0
            row[f'N(M >= {threshold:.1f})'] = rate

        threshold_rows.append(row)

    threshold_table = pd.DataFrame(threshold_rows)
    for col in threshold_table.columns[1:]:
        threshold_table[col] = threshold_table[col].map(lambda value: f'{value:.3e}')

    display(threshold_table)



### How is seismicity distributed across magnitude?

The cumulative curve shows exceedance-rate separation and upper-tail behavior, while the incremental curve shows where each file places rate across bins. Reading them together links total rate level with internal magnitude allocation.


In [ ]:
mfd_figsize = (9.0, 8.2)
mfd_marker_size = 2.5
mfd_linewidth = 1.5

if trunc_table.empty:
    print('No truncated-GR sources are available under the current selection.')
else:
    mag_mid = mfd_bundle['mag_bin_starts'] + 0.5 * mfd_bundle['bin_width']

    fig, axes = plt.subplots(2, 1, figsize=mfd_figsize, sharex=True)
    legend_handles = []

    for repo_rel in selected_files:
        stats = mfd_bundle['per_file'].get(repo_rel, {})
        n_sources = int(stats.get('n_sources', 0))

        cumulative_raw = np.asarray(stats.get('cumulative', np.zeros_like(mag_mid)), dtype=float)
        incremental_raw = np.asarray(stats.get('incremental', np.zeros_like(mag_mid)), dtype=float)

        cumulative = np.where(cumulative_raw > 0.0, cumulative_raw, np.nan)
        incremental = np.where(incremental_raw >= 0.0, incremental_raw, np.nan)

        axes[0].plot(
            mag_mid,
            cumulative,
            marker='o',
            markersize=mfd_marker_size,
            linewidth=mfd_linewidth,
            color=file_colors[repo_rel],
        )
        axes[1].plot(
            mag_mid,
            incremental,
            marker='o',
            markersize=mfd_marker_size,
            linewidth=mfd_linewidth,
            color=file_colors[repo_rel],
        )

        legend_handles.append(
            Line2D(
                [0],
                [0],
                color=file_colors[repo_rel],
                lw=2.0,
                marker='o',
                markersize=4,
                label=f"{file_labels[repo_rel]} (n={n_sources})",
            )
        )

    axes[0].set_yscale('log')
    axes[0].set_ylabel('Aggregated cumulative annual rate N(M >= m)')
    axes[0].set_title('Cumulative MFD (log scale)')
    axes[0].grid(True, which='both', linewidth=0.35, alpha=0.35)

    axes[1].set_ylabel('Aggregated incremental annual rate per bin')
    axes[1].set_xlabel('Magnitude')
    axes[1].set_title('Incremental MFD')
    axes[1].grid(True, which='both', linewidth=0.35, alpha=0.35)

    axes[0].legend(handles=legend_handles, loc='best', frameon=True)
    plt.tight_layout()



### What explains the differences between files?

Parameter medians summarize productivity (`aValue`), slope (`bValue`), onset (`minMag`), and upper-tail support (`maxMag`). The capability curve then shows how many sources in each file can still contribute above each magnitude.


In [ ]:
capability_figsize = (9.0, 4.4)

if trunc_table.empty:
    print('No truncated-GR sources are available under the current selection.')
else:
    param_grouped = (
        trunc_table.groupby('repo_rel')
        .agg(
            median_aValue=('aValue', 'median'),
            median_bValue=('bValue', 'median'),
            median_minMag=('minMag', 'median'),
            median_maxMag=('maxMag', 'median'),
            min_maxMag=('maxMag', 'min'),
            max_maxMag=('maxMag', 'max'),
        )
    )

    param_summary = pd.DataFrame(index=selected_files).join(param_grouped)
    param_summary.insert(0, 'file', [file_labels[repo_rel] for repo_rel in param_summary.index])

    for col in [
        'median_aValue',
        'median_bValue',
        'median_minMag',
        'median_maxMag',
        'min_maxMag',
        'max_maxMag',
    ]:
        param_summary[col] = param_summary[col].round(3)

    display(param_summary.reset_index(drop=True))

    capability_mag = np.asarray(mfd_bundle['mag_bin_starts'] + 0.5 * mfd_bundle['bin_width'], dtype=float)

    fig, ax = plt.subplots(figsize=capability_figsize)
    for repo_rel in selected_files:
        max_mags = trunc_table.loc[trunc_table['repo_rel'] == repo_rel, 'maxMag'].dropna().to_numpy()
        if max_mags.size == 0:
            continue

        capability_counts = np.array([(max_mags >= mag).sum() for mag in capability_mag], dtype=float)
        ax.plot(
            capability_mag,
            capability_counts,
            color=file_colors[repo_rel],
            linewidth=1.8,
            label=file_labels[repo_rel],
        )

    ax.set_xlabel('Magnitude m')
    ax.set_ylabel('Number of sources with maxMag >= m')
    ax.set_title('Source capability by magnitude')
    ax.grid(True, linewidth=0.35, alpha=0.35)
    ax.legend(loc='best', frameon=True)
    plt.tight_layout()



## 6. v1 Guardrails And Deferred Topics

- Discovery can browse the full chosen scope (including directories and unsupported files), but analysis remains file-centric and currently implements the `asm_v12e` area-source XML path only.
- `twingr.xml` remains excluded from standard comparisons.
- The post-filter workflow is intentionally rate-first: retained inventory baseline, threshold-rate table, cumulative and incremental MFD, then explanatory parameter context.
- Spatial filtering selects source membership only and does not modify source rates.
